
# 27 期权动态对冲与逐笔现金账

> **本章要解决的问题**：卖出期权后，怎样用期货对冲？为什么 Delta 中性了，账户还会每天盈亏？调仓越勤，真的越好吗？

## 本章导读

第25章讲了希腊字母，这一章把希腊字母变成**逐笔现金账**：

- 卖出一份期权，收权利金；
- 用期货做 Delta 对冲；
- 每天/每几步重新算 Delta、调仓；
- 到期结算，算总盈亏。

本章分五节：

1. 自融资组合：期权 + 期货 + 现金；
2. Delta 对冲手数与离散调仓；
3. Gamma/Theta 的盈亏归因；
4. 调仓频率与成本；
5. 逐笔现金账与对账。

### 学习目标

- 能写出一笔“卖期权 + 期货对冲”的现金流。
- 能算 Delta 对冲手数（含乘数）。
- 能模拟离散调仓，并比较不同频率的净利/波动/换手。
- 能用 Gamma/Theta 解释一步盈亏。
- 能对一张逐笔现金账。

### 先修

第24章的 Black–76；第25章的 Delta/Gamma/Theta；会 numpy。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="27 期权动态对冲与逐笔现金账" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>27 期权动态对冲与逐笔现金账｜知识图</title>
<desc>本章主线：卖期权收权利金，用期货做 Delta 对冲；离散调仓下，Gamma 与 Theta 天天换账。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a24" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao24" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag24" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">27</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">期权动态对冲与逐笔现金账</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：卖期权收权利金，用期货做 Delta 对冲；离散调仓下，Gamma 与 Theta 天天换账。</text>
<rect x="45" y="82" width="360" height="250" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="60" y="102" font-size="10.5" fill="#1B3A5C" font-weight="bold">① 期货对冲路径（示意）</text>
<path d="M 65 300 L 390 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag24)"/>
<path d="M 65 300 L 65 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag24)"/>
<path d="M 65 300 L 65 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag24)"/>
<path d="M 65 300 L 65 130" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag24)"/>
<text x="72" y="142" font-size="9" fill="#6E7B8B">Δ 手数</text>
<text x="374" y="318" font-size="9" fill="#6E7B8B">t / 调仓</text>
<path d="M 80 250 L 120 238 L 160 246 L 200 226 L 240 234 L 280 214 L 320 222 L 360 206 L 385 212" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 3"/>
<line x1="80" y1="238.6" x2="160" y2="238.6" stroke="#1B3A5C" stroke-width="2.2" stroke-linecap="round"/>
<circle cx="80" cy="238.6" r="3.6" fill="#E07B39"/>
<line x1="80" y1="238.6" x2="80" y2="300" stroke="#E07B39" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 3"/>
<line x1="160" y1="247.8" x2="240" y2="247.8" stroke="#1B3A5C" stroke-width="2.2" stroke-linecap="round"/>
<circle cx="160" cy="247.8" r="3.6" fill="#E07B39"/>
<line x1="160" y1="247.8" x2="160" y2="300" stroke="#E07B39" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 3"/>
<line x1="240" y1="257" x2="320" y2="257" stroke="#1B3A5C" stroke-width="2.2" stroke-linecap="round"/>
<circle cx="240" cy="257" r="3.6" fill="#E07B39"/>
<line x1="240" y1="257" x2="240" y2="300" stroke="#E07B39" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 3"/>
<line x1="320" y1="263.9" x2="385" y2="263.9" stroke="#1B3A5C" stroke-width="2.2" stroke-linecap="round"/>
<circle cx="320" cy="263.9" r="3.6" fill="#E07B39"/>
<line x1="320" y1="263.9" x2="320" y2="300" stroke="#E07B39" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 3"/>
<text x="82" y="166" font-size="8.8" fill="#6E7B8B">再平衡点：按最新 Δ 调仓</text>
<text x="60" y="290" font-size="9.2" fill="#E07B39" font-weight="bold">h = −Δ × 手数 × 乘数</text>
<rect x="425" y="82" width="330" height="250" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="440" y="102" font-size="10.5" fill="#1B3A5C" font-weight="bold">② 逐笔现金账（示意）</text>
<rect x="440" y="118" width="330" height="132" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="8"/>
<rect x="440" y="118" width="330" height="24" fill="#1B3A5C" rx="8"/>
<rect x="440" y="132" width="330" height="10" fill="#1B3A5C"/>
<text x="475" y="134" font-size="9" fill="#FFFFFF" text-anchor="middle">事件</text>
<text x="543" y="134" font-size="9" fill="#FFFFFF" text-anchor="middle">期权</text>
<text x="607" y="134" font-size="9" fill="#FFFFFF" text-anchor="middle">期货</text>
<text x="668" y="134" font-size="9" fill="#FFFFFF" text-anchor="middle">成本</text>
<text x="734" y="134" font-size="9" fill="#FFFFFF" text-anchor="middle">净利</text>
<text x="475" y="160" font-size="8.8" fill="#1B3A5C" text-anchor="middle">建仓</text>
<text x="543" y="160" font-size="8.8" fill="#1B3A5C" text-anchor="middle">+8,000</text>
<text x="607" y="160" font-size="8.8" fill="#1B3A5C" text-anchor="middle">0</text>
<text x="668" y="160" font-size="8.8" fill="#E07B39" text-anchor="middle">−40</text>
<text x="734" y="160" font-size="8.8" fill="#1B3A5C" text-anchor="middle">+7,960</text>
<line x1="440" y1="169" x2="770" y2="169" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="475" y="187" font-size="8.8" fill="#1B3A5C" text-anchor="middle">调仓 ×3</text>
<text x="543" y="187" font-size="8.8" fill="#1B3A5C" text-anchor="middle">+1,200</text>
<text x="607" y="187" font-size="8.8" fill="#E07B39" text-anchor="middle">−1,100</text>
<text x="668" y="187" font-size="8.8" fill="#E07B39" text-anchor="middle">−120</text>
<text x="734" y="187" font-size="8.8" fill="#E07B39" text-anchor="middle">−20</text>
<line x1="440" y1="196" x2="770" y2="196" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="475" y="214" font-size="8.8" fill="#1B3A5C" text-anchor="middle">到期结算</text>
<text x="543" y="214" font-size="8.8" fill="#E07B39" text-anchor="middle">−9,300</text>
<text x="607" y="214" font-size="8.8" fill="#1B3A5C" text-anchor="middle">+9,500</text>
<text x="668" y="214" font-size="8.8" fill="#E07B39" text-anchor="middle">−30</text>
<text x="734" y="214" font-size="8.8" fill="#1B3A5C" text-anchor="middle">+170</text>
<line x1="440" y1="223" x2="770" y2="223" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<rect x="440" y="223" width="330" height="27" fill="#E6EFF7" rx="3"/>
<text x="475" y="241" font-size="8.8" fill="#1B3A5C" text-anchor="middle" font-weight="bold">合计</text>
<text x="543" y="241" font-size="8.8" fill="#E07B39" text-anchor="middle" font-weight="bold">−100</text>
<text x="607" y="241" font-size="8.8" fill="#1B3A5C" text-anchor="middle" font-weight="bold">+8,400</text>
<text x="668" y="241" font-size="8.8" fill="#E07B39" text-anchor="middle" font-weight="bold">−190</text>
<text x="734" y="241" font-size="8.8" fill="#1B3A5C" text-anchor="middle" font-weight="bold">+8,110</text>
<line x1="440" y1="250" x2="770" y2="250" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="440" y="306" font-size="8.8" fill="#6E7B8B">Gamma 与 Theta 每天在这里交换</text>
<text x="440" y="320" font-size="8.8" fill="#1B3A5C">调仓越勤：误差↓、成本↑</text>
<rect x="40" y="344" width="168" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="124" y="354.8" font-size="9.8" fill="#1B3A5C" text-anchor="middle">自融资组合</text>
<text x="124" y="368.6" font-size="8.2" fill="#6E7B8B" text-anchor="middle">期权 + 期货 + 现金</text>
<rect x="221" y="344" width="168" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="305" y="354.8" font-size="9.8" fill="#1B3A5C" text-anchor="middle">Delta 对冲手数</text>
<text x="305" y="368.6" font-size="8.2" fill="#6E7B8B" text-anchor="middle">含乘数与符号</text>
<rect x="402" y="344" width="168" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="486" y="354.8" font-size="9.8" fill="#1B3A5C" text-anchor="middle">离散调仓</text>
<text x="486" y="368.6" font-size="8.2" fill="#6E7B8B" text-anchor="middle">越勤成本越高</text>
<rect x="583" y="344" width="168" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="667" y="354.8" font-size="9.8" fill="#1B3A5C" text-anchor="middle">逐笔现金账</text>
<text x="667" y="368.6" font-size="8.2" fill="#6E7B8B" text-anchor="middle">到期结算与对账</text>
<path d="M 208 365 L 221 365" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a24)"/>
<path d="M 389 365 L 402 365" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a24)"/>
<path d="M 570 365 L 583 365" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a24)"/>
</svg>

> **图**：本章主线：卖期权收权利金，用期货做 Delta 对冲；离散调仓下，Gamma 与 Theta 天天换账。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：期权交易员、对冲执行、风控。
- **输入**：期权持仓、期货报价、调仓频率、成本。
- **输出**：对冲手数、逐笔现金账、频率对比、残余风险。
- **在流程的哪一步**：动态对冲与执行。
- **失败模式**：手数/乘数错、忽略调仓成本、把保证金当亏损。
- **最小工作流**：持仓 → Delta 手数 → 离散调仓 → Gamma/Theta 归因 → 现金账。

---
## 1. 自融资组合：期权 + 期货 + 现金

### 1.1 三个账户

- **期权**：卖出看涨，收权利金；到期支付 $\max(F_T-K,0)$；
- **期货**：持有 $h$ 手，按价格变化逐日结算；
- **现金**：权利金、手续费、期货盈亏都进现金。

### 1.2 一手小账

期货 100→102→101，持有 0.5 手（乘数 10）：

- 第一段：$0.5\times(102-100)\times10=10$；
- 第二段：$0.5\times(101-102)\times10=-5$；
- 合计 +5。

若第二段调仓到 0.7，第二段盈亏变成 $0.7\times(-1)\times10=-7$，合计 +3。**调仓改变了后续敞口。**

### 1.3 自融资

“自融资”指调仓不额外注入资金：买期货/期权的钱，来自现金账户；现金不够，就要融资/保证金。教学模型先忽略融资成本，但账要记全。

**常见错误**：

1. 只算期权盈亏，不算期货腿。
2. 忘记乘数。
3. 把“浮盈”当“现金”。

**停下来想一想**：卖出看涨，Delta 是负的；要用期货对冲，应该买还是卖？

<details><summary>对照你的思路</summary>

卖出看涨的 Delta 为负（标的上张，期权亏）；要 Delta 中性，需要正的期货 Delta，所以**买入期货**。买入数量约等于看涨的 Delta × 手数 × 乘数 / 期货每手乘数。

</details>

---



## 2. Delta 对冲手数与离散调仓

### 2.1 手数

期权 Delta 暴露：

$$D_{\text{option}}=\Delta\times\text{lots}_{\text{option}}\times\text{mult}_{\text{option}}.$$

期货一手 Delta = $\text{mult}_{\text{futures}}$（每点）。对冲手数：

$$h=-\frac{D_{\text{option}}}{\text{mult}_{\text{futures}}}.$$

卖出看涨时 $D_{\text{option}}<0$，所以 $h>0$（买期货）。

**公式拆解**
- **符号**：$\Delta$ 每单位 Delta，lots 手数，mult 乘数，$D$ 现金 Delta（元/点）。
- **结构**：$D=\Delta\times\text{lots}\times\text{mult}$，$h=-D/\text{mult}$。
- **代入**：卖 1 手 $\Delta=0.52$、mult$=10$：$D=-5.2$，$h=0.52$ 手（取整 0 或 1）。
- **边界**：$D=0$ 时 $h=0$；$h$ 只在调仓时刻中性。
- **用途**：动态对冲第一步；不能拿每单位 Delta 当手数。

### 2.2 离散调仓

连续对冲要求每时每刻调仓；现实只能每隔一段时间调。两次调仓之间，Delta 会随标的变化，组合重新出现方向暴露。

### 2.3 代码：一条路径的 Delta 对冲



In [2]:

import numpy as np
from scipy.special import ndtr

def call_price(F, K, T, vol, r=0.0):
    if T <= 0:
        return max(F - K, 0)
    w = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * w * w) / w
    return np.exp(-r * T) * (F * ndtr(d1) - K * ndtr(d1 - w))

def call_delta(F, K, T, vol):
    if T <= 0:
        return 1.0 if F > K else 0.0
    w = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * w * w) / w
    return ndtr(d1)

F0, K, T, vol = 100.0, 100.0, 0.25, 0.22
N = 60
dt = T / N
premium = call_price(F0, K, T, vol)

rng = np.random.default_rng(2401)
z = rng.normal(size=N)
F = F0 * np.exp(np.r_[0, np.cumsum(-0.5 * vol ** 2 * dt + vol * np.sqrt(dt) * z)])

h = 0.0
pnl_futures = 0.0
for t in range(N):
    target = call_delta(F[t], K, T - t * dt, vol)
    h = target
    pnl_futures += h * (F[t + 1] - F[t])

payoff = max(F[-1] - K, 0)
print("权利金", round(premium, 4), "到期支付", round(payoff, 4))
print("期货腿盈亏", round(pnl_futures, 4))
print("净盈亏（未计成本）", round(premium - payoff + pnl_futures, 4))


权利金 4.3862 到期支付 23.4727
期货腿盈亏 18.937
净盈亏（未计成本） -0.1496



**读数与应用**：卖出看涨收权利金，到期支付可能很大；期货腿对冲掉一部分方向风险，但离散调仓会留下误差。净盈亏 = 权利金 − 支付 + 期货盈亏。

**常见错误**：

1. 用“每单位 Delta”而不是现金 Delta 算手数。
2. 调仓时用当根收盘价成交。
3. 忘记到期支付。

**停下来想一想**：为什么两次调仓之间，组合会重新出现 Delta？

<details><summary>对照你的思路</summary>

Delta 随标的和剩余期限变化；两次调仓之间，标的动了，期权 Delta 变了，但期货手数没变，所以净 Delta 不再为 0。Gamma 越大、标的动得越大，偏离越明显。

</details>

---



## 3. Gamma/Theta 的盈亏归因

### 3.1 一阶与二阶

Delta 对冲后，组合的局部损益近似：

$$dV\approx\frac12\Gamma(dF)^2+\Theta_\tau\,d\tau.$$

- 正 Gamma：标的大动赚，横盘亏 Theta；
- 负 Gamma：标的大动亏，横盘赚 Theta。

**公式拆解**
- **符号**：$\Gamma$、$\Theta_\tau$ 为曲率与时间敏感度，$dF,d\tau$ 为一步变动。
- **结构**：$dV\approx\frac12\Gamma(dF)^2+\Theta_\tau d\tau$，Delta 已对冲。
- **代入**：$\Gamma=0.036,\Theta_\tau=8.92$，$dF=1,d\tau=-1/60$：$-0.13$。
- **边界**：$dF=0$ 时 $dV<0$（买方每天损耗）。
- **用途**：解释 Delta 中性后的盈亏；$\Theta$ 符号别混。

### 3.2 实际波动 vs 隐含波动

卖出期权、Delta 对冲，赚的是“实际波动 < 隐含波动”的差；亏的是“实际波动 > 隐含波动”。这不是精确等式，但方向对：

- 实际波动小 → 空头 Gamma 赚 Theta；
- 实际波动大 → 空头 Gamma 亏 Gamma。

### 3.3 代码：一步归因



In [3]:

F_t, F_next = 100.0, 101.0
T_t, step = 0.25, 1 / 60

V0 = call_price(F_t, K, T_t, vol)
V1 = call_price(F_next, K, T_t - step, vol)

h = 0.01
delta = (call_price(F_t + h, K, T_t, vol) - call_price(F_t - h, K, T_t, vol)) / (2 * h)
gamma = (call_price(F_t + h, K, T_t, vol) - 2 * V0 + call_price(F_t - h, K, T_t, vol)) / h ** 2
theta_tau = (call_price(F_t, K, T_t - step, vol) - V0) / step

dF = F_next - F_t
approx = delta * dF + 0.5 * gamma * dF ** 2 + theta_tau * step

print("实际变化", round(V1 - V0, 5))
print("Delta/Gamma/Theta_tau", round(delta, 4), round(gamma, 4), round(theta_tau, 4))
print("近似", round(approx, 5), "残差", round((V1 - V0) - approx, 5))


实际变化 0.39124
Delta/Gamma/Theta_tau 0.5219 0.0362 -8.9151
近似 0.39145 残差 -0.00021



**读数与应用**：一步之内，Delta 项、Gamma 项、时间项相加，应接近实际变化；残差来自高阶项和离散误差。

**常见错误**：

1. 把 $\Theta_\tau$（对剩余期限）和 $\Theta_t$（对日历时间）符号搞反。
2. 用年化 Theta 乘一步的时间，忘记单位。
3. 标的大动时只用 Delta，忽略 Gamma。

**停下来想一想**：卖出期权、Delta 对冲，最怕什么？

<details><summary>对照你的思路</summary>

最怕“实际波动远大于隐含波动”：负 Gamma 在标的大动时亏，Theta 收的钱不够补。跳空更糟，因为 Gamma 对冲来不及调整。卖方策略的尾部风险就在这里。

</details>

---



## 4. 调仓频率与成本

### 4.1 越勤越好吗

调仓越勤：

- 方向暴露越小（跟踪误差小）；
- 交易成本越高（手续费、价差、冲击）。

存在一个“最优频率”，取决于 Gamma、成本、流动性。

### 4.2 模拟

用同一组路径，比较每 1/5/20 步调仓的净盈亏与换手。

### 4.3 代码：频率对比



In [4]:

def hedge_one(z_path, every):
    F = F0 * np.exp(np.r_[0, np.cumsum(-0.5 * vol ** 2 * dt + vol * np.sqrt(dt) * z_path)])
    h = 0.0
    pnl = 0.0
    turnover = 0.0
    for t in range(N):
        if t % every == 0:
            target = call_delta(F[t], K, T - t * dt, vol)
            turnover += abs(target - h)
            h = target
        pnl += h * (F[t + 1] - F[t])
    payoff = max(F[-1] - K, 0)
    return premium - payoff + pnl, turnover

rng = np.random.default_rng(2402)
z_paths = rng.normal(size=(500, N))

for every in [1, 5, 20]:
    net, turn = np.array([hedge_one(z, every) for z in z_paths]).T
    print("每", every, "步调仓：净盈亏均值", round(net.mean(), 4),
          "标准差", round(net.std(ddof=1), 4), "平均换手", round(turn.mean(), 2))


每 1 步调仓：净盈亏均值 -0.0126 标准差 0.4783 平均换手 2.9
每 5 步调仓：净盈亏均值 -0.0341 标准差 1.0414 平均换手 1.54
每 20 步调仓：净盈亏均值 -0.1102 标准差 2.0842 平均换手 0.93



**读数与应用**：调仓越勤，净盈亏的波动通常越小，但换手/成本越高；净均值可能下降。真实选择要加成本项。

**常见错误**：

1. 只比均值，不比波动/尾部。
2. 忽略调仓成本。
3. 用不同路径比较不同频率（样本不可比）。

**停下来想一想**：为什么“调仓越勤，波动越小”？

<details><summary>对照你的思路</summary>

调仓越勤，Delta 偏离越小，组合的方向暴露越少，跟踪误差/盈亏波动下降。但每次调仓有成本，净收益不一定更高。

</details>

---



## 5. 逐笔现金账与对账

### 5.1 一张账

| 项目 | 金额 |
|---|---:|
| 权利金收入 | +premium |
| 到期支付 | −payoff |
| 期货腿盈亏 | ±pnl |
| 手续费/滑点 | −cost |
| 净额 | 合计 |

### 5.2 对账

- 权利金按“每吨/每点 × 乘数”；
- 期货腿按“手数 × 乘数 × 价格变化”；
- 到期支付按“内在价值 × 乘数”；
- 手续费按双边；
- 保证金/利息另记。

### 5.3 代码：现金账



In [5]:

import pandas as pd

cost = 0.0
ledger = pd.DataFrame({
    "项目": ["权利金收入", "到期支付", "期货腿盈亏", "手续费", "净额"],
    "金额": [premium, -payoff, pnl_futures, -cost, premium - payoff + pnl_futures - cost],
})
print(ledger.round(4).to_string(index=False))


   项目       金额
权利金收入   4.3862
 到期支付 -23.4727
期货腿盈亏  18.9370
  手续费  -0.0000
   净额  -0.1496



**读数与应用**：这张账只含期权/期货/手续费；真实账户还有保证金、利息、强平。对账时要把每一项落到“哪一腿、哪一笔”。

**常见错误**：

1. 权利金不乘乘数。
2. 期货腿用“点数”当“元”。
3. 忘记到期支付。
4. 把保证金当亏损。

**停下来想一想**：为什么“保证金增加”不是“亏损”？

<details><summary>对照你的思路</summary>

保证金是履约保证/冻结资金，不是已经损失的钱；它会影响可用资金和强平风险，但不直接进损益。亏损来自价格变化/支付/费用。账要分“权益”和“可用资金”。

</details>

---



## 本章小结

### 一张表

| 概念 | 公式/做法 | 用途 |
|---|---|---|
| 期权 Delta 暴露 | $\Delta\times\text{lots}\times\text{mult}$ | 对冲 |
| 期货手数 | $-D/\text{mult}$ | Delta 中性 |
| 局部损益 | $\frac12\Gamma(dF)^2+\Theta_\tau d\tau$ | 归因 |
| 调仓频率 | 跟踪误差 vs 成本 | 选择 |
| 现金账 | 权利金−支付+期货盈亏−费用 | 对账 |

### 八句话

1. 卖出看涨 Delta 为负，要用买期货对冲。
2. 手数要乘乘数。
3. 两次调仓之间，Delta 会漂移。
4. 正 Gamma 怕横盘，负 Gamma 怕大动。
5. 卖方赚 Theta，怕实际波动大于隐含。
6. 调仓越勤，跟踪误差小、成本高。
7. 权利金、支付、期货盈亏、费用要逐项记。
8. 保证金不是亏损。

---

## 本章习题

**1.** 卖出看涨，Delta −0.5，10 手，乘数 10。期货乘数 10。对冲需要多少手？

**2.** 期货 100→102→101，持有 0.5 手，乘数 10。求两段合计盈亏。

**3.** 正 Gamma 的组合，标的大动/横盘分别怎样？

**4.** 负 Gamma 的卖方，最怕什么？

**5.** 调仓越勤，跟踪误差和成本分别怎样变？

**6.** 权利金 6.97、到期支付 10、期货盈亏 4、费用 1。求净额（每单位）。

**7.** 为什么保证金增加不是亏损？

**8.** 为什么两次调仓之间会出现 Delta？

<details><summary>参考答案</summary>

**1.** 期权 Delta 暴露 $-0.5\times10\times10=-50$；期货一手 10，需 $-(-50)/10=5$ 手。

**2.** $0.5\times2\times10+0.5\times(-1)\times10=10-5=5$。

**3.** 正 Gamma：大动赚，横盘亏 Theta。

**4.** 实际波动/跳空远大于隐含波动。

**5.** 跟踪误差下降，成本上升。

**6.** $6.97-10+4-1=-0.03$。

**7.** 保证金是冻结资金/履约保证，不是已实现损失；影响可用资金，不进损益。

**8.** 标的动了，期权 Delta 变了，期货手数没变，净 Delta 不再为 0。

</details>

---

## 参考资料

- Hull, J. C. *Options, Futures, and Other Derivatives*（动态对冲）。
- Taleb, N. N. *Dynamic Hedging*（离散对冲/尾部）。
